# LeetCode 3141 — Maximum Hamming Distances bằng Python

- Link: https://leetcode.com/problems/maximum-hamming-distances/
- Độ khó: **Hard** (bài **Premium**)
- Ghi chú nguồn: đề, ví dụ, ràng buộc và chữ ký hàm được **đối chiếu** từ https://leetcode.doocs.org/en/lc/3141/ (không có tài khoản Premium để xem trực tiếp trên LeetCode — nên đối chiếu lại nếu có).
- Chủ đề: **Đảo bài toán** — Hamming lớn nhất tới tập `= m − (Hamming nhỏ nhất từ phần bù của x tới tập)`; tính khoảng cách nhỏ nhất tới tập cho **mọi** mask bằng **BFS đa nguồn trên siêu lập phương** hoặc **DP theo từng chiều**, `O(m · 2^m)`

Skeleton (theo doocs):

```python
class Solution:
    def maxHammingDistances(self, nums: List[int], m: int) -> List[int]:
```

## 1. Mục tiêu học

1. Khoảng cách Hamming `H(x, y) = popcount(x XOR y)`; với `m` bit, `H(x, y) = m − H(x̄, y)` trong đó `x̄ = x XOR (2^m − 1)`.
2. Do đó `max_y H(x, y) = m − min_y H(x̄, y)` — bài **lớn nhất** thành bài **nhỏ nhất** tới tập.
3. `dist[z] = min_{y ∈ nums} H(z, y)` cho mọi `z ∈ [0, 2^m)`: BFS đa nguồn trên đồ thị siêu lập phương (cạnh = lật 1 bit), `O(m · 2^m)`.
4. Hoặc DP theo chiều: lần lượt với mỗi bit `b`, `dist[z] = min(dist[z], dist[z ^ 2^b] + 1)` — một lượt mỗi bit là đủ.
5. Kiểm chứng bằng brute `O(n² )`.

## 2. Đề bài & ví dụ

Cho mảng `nums` và số nguyên `m`, mọi `0 ≤ nums[i] < 2^m`. Trả về mảng `answer` cùng độ dài, `answer[i]` = khoảng cách Hamming **lớn nhất** giữa `nums[i]` và một phần tử **khác** `nums[j]` (`j ≠ i`) của mảng. Khoảng cách Hamming = số vị trí bit khác nhau (thêm số 0 đứng đầu nếu cần).

**Ví dụ 1:** `nums = [9,12,9,11]`, `m = 4` → `[2,3,2,3]`.

**Ví dụ 2:** `nums = [3,4,6,10]`, `m = 4` → `[3,3,2,3]`.

**Ràng buộc:** `1 ≤ m ≤ 17`; `2 ≤ nums.length ≤ 2^m`; `0 ≤ nums[i] < 2^m`.

In [1]:
from typing import List
from collections import deque
import random, time

examples = [([9, 12, 9, 11], 4, [2, 3, 2, 3]), ([3, 4, 6, 10], 4, [3, 3, 2, 3])]
for a, m, exp in examples:
    print([format(x, f'0{m}b') for x in a], '-> kỳ vọng', exp)

['1001', '1100', '1001', '1011'] -> kỳ vọng [2, 3, 2, 3]
['0011', '0100', '0110', '1010'] -> kỳ vọng [3, 3, 2, 3]


## 3. Kiến thức nền

### 3.1. Hamming và phần bù

Lật **mọi** bit của `x` thì mỗi vị trí "khác" thành "giống" và ngược lại ⇒ `H(x̄, y) = m − H(x, y)`.

In [2]:
def H(x, y):
    return bin(x ^ y).count('1')

m = 5; full = (1 << m) - 1
for x, y in [(9, 12), (3, 4), (0, 31), (21, 21)]:
    print(f'H({x}, {y}) = {H(x, y)} | m − H(x̄, y) = {m - H(x ^ full, y)}')

H(9, 12) = 2 | m − H(x̄, y) = 2
H(3, 4) = 3 | m − H(x̄, y) = 3
H(0, 31) = 5 | m − H(x̄, y) = 5
H(21, 21) = 0 | m − H(x̄, y) = 0


### 3.2. Phần tử "khác" có quan trọng không?

Đề yêu cầu `j ≠ i`. Nếu ta lấy max trên **mọi** `j` (kể cả `j = i`), phần tử chính nó góp `H(x, x) = 0` — không bao giờ lớn hơn một giá trị hợp lệ khác (luôn `≥ 0`), và vì `n ≥ 2` luôn có ít nhất một `j ≠ i`. Vậy có thể bỏ qua điều kiện `j ≠ i`. (Hai phần tử bằng nhau vẫn là hai phần tử khác nhau — cho khoảng cách 0.)

## 4. Ý tưởng nền tảng / cách nghĩ trực giác

So từng cặp là `O(n²)` với `n` tới `2^{17} ≈ 1.3·10⁵` — quá chậm. Nhưng không gian giá trị chỉ có `2^m ≤ 1.3·10⁵` điểm: tính trước "khoảng cách tới tập gần nhất" cho **mọi** điểm của siêu lập phương, rồi mỗi truy vấn tra `O(1)` tại phần bù.

Ví dụ 1: `x = 12 = 1100`, `x̄ = 0011`; phần tử gần `0011` nhất là `1011` (khác 1 bit) ⇒ đáp án `4 − 1 = 3`.

## 5. Cách 1 — Brute Force: mọi cặp

In [3]:
def brute(nums, m):
    n = len(nums)
    return [max(H(nums[i], nums[j]) for j in range(n) if j != i) for i in range(n)]

for a, m, exp in examples:
    print('brute:', brute(a, m), '| kỳ vọng', exp)

brute: [2, 3, 2, 3] | kỳ vọng [2, 3, 2, 3]
brute: [3, 3, 2, 3] | kỳ vọng [3, 3, 2, 3]


**Độ phức tạp:** `O(n²)` — với `n = 2^{17}` là `1.7·10^{10}`.

## 6. Ý tưởng thuật toán / bước đột phá

```
dist[z] = 0 với z ∈ nums; BFS lan bằng cách lật từng bit
answer[i] = m − dist[nums[i] XOR (2^m − 1)]
```

## 7. Các cách tiếp theo

### 7.1. Cách 2 — BFS đa nguồn trên siêu lập phương (lời giải chính)

In [4]:
def bfs_solution(nums, m):
    N = 1 << m; dist = [-1] * N; q = deque()
    for x in nums:
        if dist[x] < 0:
            dist[x] = 0; q.append(x)
    while q:
        x = q.popleft(); d = dist[x] + 1
        for b in range(m):
            y = x ^ (1 << b)
            if dist[y] < 0:
                dist[y] = d; q.append(y)
    full = N - 1
    return [m - dist[x ^ full] for x in nums]

for a, m, exp in examples:
    print('BFS:', bfs_solution(a, m), '| kỳ vọng', exp)

BFS: [2, 3, 2, 3] | kỳ vọng [2, 3, 2, 3]
BFS: [3, 3, 2, 3] | kỳ vọng [3, 3, 2, 3]


### 7.2. Cách 3 — DP theo từng chiều (không cần hàng đợi)

Đường ngắn nhất trên siêu lập phương lật mỗi bit **tối đa một lần**, thứ tự tuỳ ý. Xử lý các bit lần lượt `b = 0, 1, …, m−1`: sau khi xử lý tập bit `B`, `dist[z]` = khoảng cách nhỏ nhất tới tập khi chỉ được lật các bit trong `B` ⇒ sau `m` lượt là khoảng cách thật.

In [5]:
def dim_dp(nums, m):
    N = 1 << m; INF = m + 1
    dist = [INF] * N
    for x in nums:
        dist[x] = 0
    for b in range(m):
        bit = 1 << b
        for z in range(N):
            if z & bit:
                a, c = dist[z], dist[z ^ bit]
                if c + 1 < a:
                    dist[z] = c + 1
                elif a + 1 < c:
                    dist[z ^ bit] = a + 1
    full = N - 1
    return [m - dist[x ^ full] for x in nums]

for a, m, exp in examples:
    print('DP theo chiều:', dim_dp(a, m), '| kỳ vọng', exp)

DP theo chiều: [2, 3, 2, 3] | kỳ vọng [2, 3, 2, 3]
DP theo chiều: [3, 3, 2, 3] | kỳ vọng [3, 3, 2, 3]


### 7.3. Cách 4 — Brute trên **giá trị khác nhau** + `int.bit_count`

Khi có nhiều phần tử trùng, chỉ cần so các giá trị khác nhau `D` (`O(D²)`), rồi ánh xạ lại. Hữu ích khi `m` lớn nhưng `D` nhỏ (không phải trường hợp xấu nhất của bài).

In [6]:
def distinct_brute(nums, m):
    vals = sorted(set(nums))
    if len(vals) == 1:
        return [0] * len(nums)
    best = {x: max((x ^ y).bit_count() for y in vals) for x in vals}
    return [best[x] for x in nums]

for a, m, exp in examples:
    print('giá trị khác nhau:', distinct_brute(a, m), '| kỳ vọng', exp)

giá trị khác nhau: [2, 3, 2, 3] | kỳ vọng [2, 3, 2, 3]
giá trị khác nhau: [3, 3, 2, 3] | kỳ vọng [3, 3, 2, 3]


(Với mỗi `x`, `max` trên `vals` gồm cả `x` — góp 0, vô hại như mục 3.2; khi mọi phần tử bằng nhau, đáp án là 0.)

## 8. TRACE từng bước

Trace BFS trên ví dụ 2 (`m = 4`): in các tầng BFS và khoảng cách tại phần bù.

In [7]:
def trace(nums, m):
    N = 1 << m; dist = [-1] * N; layer = []
    for x in nums:
        if dist[x] < 0:
            dist[x] = 0; layer.append(x)
    d = 0
    while layer:
        print(f'tầng {d}: {[format(x, f"0{m}b") for x in sorted(layer)]}')
        nxt = []
        for x in layer:
            for b in range(m):
                y = x ^ (1 << b)
                if dist[y] < 0:
                    dist[y] = d + 1; nxt.append(y)
        layer = nxt; d += 1
    full = N - 1
    for x in nums:
        print(f'x = {format(x, f"0{m}b")}, x̄ = {format(x ^ full, f"0{m}b")}, dist(x̄) = {dist[x ^ full]} -> {m - dist[x ^ full]}')

trace(*examples[1][:2])

tầng 0: ['0011', '0100', '0110', '1010']
tầng 1: ['0000', '0001', '0010', '0101', '0111', '1000', '1011', '1100', '1110']
tầng 2: ['1001', '1101', '1111']
x = 0011, x̄ = 1100, dist(x̄) = 1 -> 3
x = 0100, x̄ = 1011, dist(x̄) = 1 -> 3
x = 0110, x̄ = 1001, dist(x̄) = 2 -> 2
x = 1010, x̄ = 0101, dist(x̄) = 1 -> 3


## 9. Code nộp LeetCode

In [8]:
class Solution:
    def maxHammingDistances(self, nums: List[int], m: int) -> List[int]:
        N = 1 << m
        dist = [-1] * N
        layer = []
        for x in nums:
            if dist[x] < 0:
                dist[x] = 0
                layer.append(x)
        d = 1
        bits = [1 << b for b in range(m)]
        while layer:
            nxt = []
            for x in layer:
                for bit in bits:
                    y = x ^ bit
                    if dist[y] < 0:
                        dist[y] = d
                        nxt.append(y)
            layer = nxt
            d += 1
        full = N - 1
        return [m - dist[x ^ full] for x in nums]

for a, m, exp in examples:
    got = Solution().maxHammingDistances(a, m)
    print(got, 'OK' if got == exp else 'SAI')

[2, 3, 2, 3] OK
[3, 3, 2, 3] OK


## 10. Vì sao thuật toán đúng

1. **Đảo bài toán:** `H(x, y) = m − H(x̄, y)` ⇒ `max_y H(x, y) = m − min_y H(x̄, y)`.
2. **BFS:** đồ thị có đỉnh `[0, 2^m)`, cạnh nối hai mask khác đúng một bit; khoảng cách đồ thị giữa `z` và `y` bằng `H(z, y)` (mỗi bước sửa đúng một bit khác). BFS đa nguồn từ tập `nums` cho `dist[z] = min_y H(z, y)`. Đồ thị liên thông nên mọi `dist` được gán.
3. **`j ≠ i`:** bỏ qua được (mục 3.2).

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Brute | `O(n²)` | `O(n)` |
| 2. BFS siêu lập phương | `O(m · 2^m + n)` | `O(2^m)` |
| 3. DP theo chiều | `O(m · 2^m + n)` | `O(2^m)` |
| 4. Brute giá trị khác nhau | `O(D² + n)` | `O(D)` |

`m = 17`: `17 · 131072 ≈ 2.2·10⁶` bước.

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [9]:
impls = {'bfs': bfs_solution, 'dim_dp': dim_dp, 'distinct': distinct_brute,
         'submit': lambda a, m: Solution().maxHammingDistances(a, m)}
edge = [([0, 1], 1, [1, 1]), ([5, 5], 3, [0, 0]), ([0, 7], 3, [3, 3]), ([0, 1, 2, 3], 2, [2, 2, 2, 2]),
        ([6, 6, 1], 3, [3, 3, 3]), ([0, 0, 0, 1], 5, [1, 1, 1, 1])]
for a, m, exp in examples + edge:
    assert brute(a, m) == exp, (a, m, brute(a, m))
    for name, f in impls.items():
        assert f(a, m) == exp, (name, a, m)
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


Giải thích: `[5, 5]` — phần tử khác có cùng giá trị ⇒ khoảng cách 0. `[6, 6, 1]`: `110` và `001` khác cả 3 bit.

### 12.2. Đối chiếu ngẫu nhiên với brute

In [10]:
rng = random.Random(3141)
cnt = 0
for _ in range(400):
    m = rng.randint(1, 8); n = rng.randint(2, min(1 << m, 30))
    a = [rng.randrange(1 << m) for _ in range(n)]
    exp = brute(a, m)
    for name, f in impls.items():
        assert f(a, m) == exp, (name, a, m)
    cnt += 1
print('Khớp brute trên', cnt, 'bộ ngẫu nhiên.')

Khớp brute trên 400 bộ ngẫu nhiên.


### 12.3. Ca lớn `m = 17`, `n = 2^{17}`

In [11]:
m = 17; N = 1 << m
cases = {
    'ngẫu nhiên n = 2^17': [rng.randrange(N) for _ in range(N)],
    'chỉ 2 giá trị': [rng.choice([0, 12345]) for _ in range(N)],
    'mọi giá trị': list(range(N)),
}
for name, a in cases.items():
    t = time.time(); x = Solution().maxHammingDistances(a, m); t1 = time.time() - t
    t = time.time(); y = dim_dp(a, m); t2 = time.time() - t
    assert x == y
    print(f'{name:20s}: max = {max(x)}, min = {min(x)} | nộp {t1:.2f}s | DP theo chiều {t2:.2f}s')
assert Solution().maxHammingDistances(list(range(N)), m) == [m] * N
print('Mọi giá trị có mặt: phần bù luôn có ⇒ mọi đáp án = m.')

ngẫu nhiên n = 2^17 : max = 17, min = 16 | nộp 0.10s | DP theo chiều 0.11s
chỉ 2 giá trị       : max = 6, min = 6 | nộp 0.08s | DP theo chiều 0.10s


mọi giá trị         : max = 17, min = 17 | nộp 0.08s | DP theo chiều 0.10s
Mọi giá trị có mặt: phần bù luôn có ⇒ mọi đáp án = m.


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Tra `dist` tại `x` thay vì tại phần bù `x̄`

In [12]:
def wrong_no_complement(nums, m):
    N = 1 << m; dist = [-1] * N; q = deque()
    for x in nums:
        if dist[x] < 0:
            dist[x] = 0; q.append(x)
    while q:
        x = q.popleft()
        for b in range(m):
            y = x ^ (1 << b)
            if dist[y] < 0:
                dist[y] = dist[x] + 1; q.append(y)
    return [m - dist[x] for x in nums]                                   # SAI: dist[x] = 0

print('sai :', wrong_no_complement(*examples[0][:2]), '| đúng:', Solution().maxHammingDistances(*examples[0][:2]))

sai : [4, 4, 4, 4] | đúng: [2, 3, 2, 3]


**Vì sao sai:** `x` thuộc tập nên `dist[x] = 0` ⇒ luôn trả `m`. Khoảng cách **lớn nhất** tới tập là `m` trừ khoảng cách **nhỏ nhất** từ **phần bù**.

### Lỗi 2 — Dùng trie "max XOR" (tối đa giá trị XOR thay vì số bit 1)

In [13]:
def wrong_max_xor(nums, m):
    root = {}
    for x in nums:
        node = root
        for b in range(m - 1, -1, -1):
            node = node.setdefault(x >> b & 1, {})
    res = []
    for x in nums:
        node = root; y = 0
        for b in range(m - 1, -1, -1):
            want = 1 - (x >> b & 1)
            if want in node:
                node = node[want]; y |= want << b
            else:
                bit = 1 - want; node = node[bit]; y |= bit << b
        res.append(H(x, y))                                                 # SAI
    return res

a = [4, 3, 0]
print('sai :', wrong_max_xor(a, 3), '| đúng:', Solution().maxHammingDistances(a, 3))

sai : [3, 3, 1] | đúng: [3, 3, 2]


**Vì sao sai:** với `x = 0`, XOR lớn nhất là với `4 = 100` (giá trị 4, một bit) nhưng Hamming lớn nhất là với `3 = 011` (hai bit). Tham lam bit cao ưu tiên **giá trị**, không phải **số bit khác**.

### Lỗi 3 — Ước lượng bằng `min(m, popcount(x) + max popcount(y))`

In [14]:
def wrong_popcount_bound(nums, m):
    mx = max(bin(y).count('1') for y in nums)
    return [min(m, bin(x).count('1') + mx) for x in nums]                # SAI: chỉ là cận trên

a = [3, 3]
print('sai :', wrong_popcount_bound(a, 2), '| đúng:', Solution().maxHammingDistances(a, 2))

sai : [2, 2] | đúng: [0, 0]


**Vì sao sai:** `H(x, y) ≤ popcount(x) + popcount(y)` chỉ là cận trên — đạt khi `x`, `y` không chung bit 1. Hai số bằng nhau có Hamming 0.

### Lỗi 4 — DP theo chiều chỉ cập nhật **một chiều** (`z` có bit từ `z` không bit)

In [15]:
def wrong_one_direction(nums, m):
    N = 1 << m; dist = [m + 1] * N
    for x in nums:
        dist[x] = 0
    for b in range(m):
        bit = 1 << b
        for z in range(N):
            if z & bit:
                dist[z] = min(dist[z], dist[z ^ bit] + 1)                 # SAI: quên chiều ngược lại
    full = N - 1
    return [m - dist[x ^ full] for x in nums]

a = [1, 3]
print('sai :', wrong_one_direction(a, 2), '| đúng:', Solution().maxHammingDistances(a, 2))

sai : [-1, -1] | đúng: [1, 1]


**Vì sao sai:** với `[1, 3]` (`m = 2`), phần bù của `1 = 01` là `10`; muốn tới tập phải **tắt** bit 1 rồi bật bit 0 — DP một chiều không bao giờ tắt bit nên `dist[10]` vẫn là "vô cực" `m + 1` ⇒ đáp án âm. Lật bit có thể **tắt** cũng như **bật**; chỉ lan từ mask không có bit sang mask có bit là "khoảng cách tới tập con" (kiểu SOS DP), không phải Hamming. Phải cập nhật cả hai chiều.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Khoảng cách Hamming lớn nhất" ⇒ phần bù + khoảng cách **nhỏ nhất**.
- Không gian giá trị `2^m` nhỏ ⇒ BFS đa nguồn trên siêu lập phương.
- Đường đi trên siêu lập phương: mỗi chiều tối đa một lần ⇒ DP từng chiều.

## 15. Biến thể của bài

1. Hamming **nhỏ nhất** tới phần tử khác ⇒ cần phân biệt chính nó: BFS lưu **hai** nguồn gần nhất khác nhau (hoặc đếm số lần xuất hiện).
2. Đếm số `j` đạt khoảng cách lớn nhất ⇒ BFS mang theo số đếm (cẩn thận đếm trùng đường).
3. Truy vấn online thêm phần tử ⇒ BFS tăng dần (chỉ cập nhật các mask mà khoảng cách giảm).
4. Với `m` lớn (64 bit) và `n` nhỏ ⇒ quay lại brute hoặc các kỹ thuật xấp xỉ.

In [16]:
# Biến thể 1: Hamming NHỎ NHẤT tới phần tử khác (j != i) — BFS giữ 2 nguồn gần nhất khác nhau
def min_hamming_other(nums, m):
    from collections import Counter
    cnt = Counter(nums); N = 1 << m
    best = [[] for _ in range(N)]             # tối đa 2 cặp (khoảng cách, nguồn) với nguồn khác nhau
    q = deque()
    for x in cnt:
        best[x] = [(0, x)]; q.append((x, x, 0))
    while q:
        z, src, d = q.popleft()
        for b in range(m):
            y = z ^ (1 << b)
            cur = best[y]
            if len(cur) < 2 and all(s != src for _, s in cur):
                cur.append((d + 1, src)); q.append((y, src, d + 1))
    res = []
    for x in nums:
        if cnt[x] >= 2:
            res.append(0)
        else:
            res.append(next(dd for dd, s in best[x] if s != x))
    return res

def min_brute(nums, m):
    n = len(nums)
    return [min(H(nums[i], nums[j]) for j in range(n) if j != i) for i in range(n)]

for _ in range(300):
    m = rng.randint(1, 7); a = [rng.randrange(1 << m) for _ in range(rng.randint(2, 12))]
    assert min_hamming_other(a, m) == min_brute(a, m)
print('Biến thể "Hamming nhỏ nhất tới phần tử khác": khớp brute.')

Biến thể "Hamming nhỏ nhất tới phần tử khác": khớp brute.


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Ghi chú |
|---|---|---|---|
| 1 | Mọi cặp | `O(n²)` | Oracle |
| 2 | BFS siêu lập phương | `O(m 2^m)` | **Nộp bài** |
| 3 | DP theo chiều | `O(m 2^m)` | Không cần hàng đợi |
| 4 | Brute giá trị khác nhau | `O(D²)` | Khi ít giá trị |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `nums = [1, 2]`, `m = 2`.

**Bài 2 (tính tay):** `nums = [0, 5, 6]`, `m = 3`.

**Bài 3 (điền TODO):** trả về luôn một chỉ số `j ≠ i` đạt khoảng cách lớn nhất cho mỗi `i`.

**Bài 4 (tự viết lại):** khoảng cách Hamming lớn nhất giữa **mọi** cặp trong mảng (một số).

**Bài 5 (mini challenge):** cho thêm `q` truy vấn giá trị `v` (không nằm trong mảng): Hamming lớn nhất từ `v` tới mảng.

In [17]:
def argmax_partner_todo(nums, m):
    # TODO
    raise NotImplementedError

def global_max_todo(nums, m):
    # TODO
    raise NotImplementedError

def queries_todo(nums, m, qs):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) `01` và `10`. (2) Khác cả 2 bit. (3) `[2, 2]`.

**Bài 2.** (1) `000, 101, 110`. (2) `H(0,5) = 2, H(0,6) = 2, H(5,6) = 2`. (3) `[2, 2, 2]`.

**Bài 3.** (1) BFS lưu thêm `src[z]` = một phần tử của tập gần `z` nhất. (2) Đối tác của `x` là `src[x̄]`. (3) Nếu `src[x̄]` có cùng giá trị nhưng đó chính là chỉ số `i` — chọn chỉ số khác cùng giá trị nếu có, còn không thì… vẫn đúng khoảng cách nhưng cần chỉ số khác: dùng chỉ số bất kỳ `j ≠ i` có giá trị đó (nếu giá trị chỉ xuất hiện một lần và là chính `x`, thì khoảng cách = 0 = m − m, chọn `j` bất kỳ).

**Bài 4.** (1) `max(answer)`. (2) Hoặc `m − min_x dist[x̄]`. (3) Kiểm brute.

**Bài 5.** (1) `dist` đã tính cho mọi mask. (2) Đáp án `m − dist[v̄]`. (3) `O(1)` mỗi truy vấn.

## 19. Đáp án tham khảo cho bài tập

In [18]:
# Bài 1, 2, 4, 5
assert Solution().maxHammingDistances([1, 2], 2) == [2, 2]
assert Solution().maxHammingDistances([0, 5, 6], 3) == [2, 2, 2]

def global_max(nums, m):
    return max(Solution().maxHammingDistances(nums, m))

def queries(nums, m, qs):
    N = 1 << m; dist = [-1] * N; q = deque()
    for x in nums:
        if dist[x] < 0:
            dist[x] = 0; q.append(x)
    while q:
        x = q.popleft()
        for b in range(m):
            y = x ^ (1 << b)
            if dist[y] < 0:
                dist[y] = dist[x] + 1; q.append(y)
    return [m - dist[v ^ (N - 1)] for v in qs]

for _ in range(200):
    m = rng.randint(1, 7); a = [rng.randrange(1 << m) for _ in range(rng.randint(2, 12))]
    assert global_max(a, m) == max(H(x, y) for i, x in enumerate(a) for j, y in enumerate(a) if i != j)
    qs = [rng.randrange(1 << m) for _ in range(5)]
    assert queries(a, m, qs) == [max(H(v, y) for y in a) for v in qs]
print('Bài 1: [2, 2] | Bài 2: [2, 2, 2] | Bài 4, 5 khớp brute.')

Bài 1: [2, 2] | Bài 2: [2, 2, 2] | Bài 4, 5 khớp brute.


In [19]:
# Bài 3: chỉ số đối tác
def argmax_partner(nums, m):
    N = 1 << m; src = [-1] * N; dist = [-1] * N; q = deque()
    for i, x in enumerate(nums):
        if dist[x] < 0:
            dist[x] = 0; src[x] = i; q.append(x)
    while q:
        x = q.popleft()
        for b in range(m):
            y = x ^ (1 << b)
            if dist[y] < 0:
                dist[y] = dist[x] + 1; src[y] = src[x]; q.append(y)
    pos = {}
    for i, x in enumerate(nums):
        pos.setdefault(x, []).append(i)
    res = []
    for i, x in enumerate(nums):
        j = src[x ^ (N - 1)]
        if j == i:                                   # cùng chỉ số: chọn chỉ số khác cùng giá trị, hoặc bất kỳ
            others = [t for t in pos[x] if t != i]
            j = others[0] if others else (0 if i else 1)
        res.append(j)
    return res

for _ in range(300):
    m = rng.randint(1, 7); a = [rng.randrange(1 << m) for _ in range(rng.randint(2, 12))]
    js = argmax_partner(a, m); best = brute(a, m)
    assert all(j != i and H(a[i], a[j]) == best[i] for i, j in enumerate(js))
print('Bài 3: chỉ số đối tác hợp lệ (j ≠ i) và đạt khoảng cách lớn nhất. Ví dụ 1 ->', argmax_partner(*examples[0][:2]))

Bài 3: chỉ số đối tác hợp lệ (j ≠ i) và đạt khoảng cách lớn nhất. Ví dụ 1 -> [1, 3, 1, 1]


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
dist = [-1] * (1 << m); layer = list(set(nums)); dist[x] = 0 cho x trong nums
BFS: lật từng bit, gán dist khi lần đầu gặp
answer = [m - dist[x ^ ((1 << m) - 1)] for x in nums]
```

**Checklist:**

- [ ] Tra tại **phần bù** `x ^ (2^m − 1)`?
- [ ] Dùng **số bit khác** (popcount), không phải giá trị XOR?
- [ ] BFS đa nguồn khởi tạo mọi phần tử (khử trùng)?
- [ ] Đã hiểu vì sao `j ≠ i` không cần xử lý riêng?